In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

MEASUREMENTS_DIR = Path("../../test_results/0_measurements")
OUTPUT_DIR = MEASUREMENTS_DIR / "errors"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

reference = pd.read_csv(MEASUREMENTS_DIR / "reference_measurements.csv").set_index("image_id")

models = [
    "simplebaseline_res50", "hrnet_w32", "litehrnet18",
    "rtmpose_m", "vitpose_b", "mspn_4stage",
]
units = {
    "ca_deg": "CA",
    "hva_deg": "HVA",
    "nmp_percent": "NMP",
    "si_percent": "SI",
    "csi_percent": "CSI",
}

In [2]:
summary_rows = []

for model in models:
    prediction = pd.read_csv(MEASUREMENTS_DIR / f"{model}_measurements.csv").set_index("image_id").loc[reference.index]

    error_tables = []
    for metric, unit in units.items():
        error = prediction[metric] - reference[metric]
        error_tables.append(pd.DataFrame({
            "sample_name": reference["sample_name"],
            "metric": metric,
            "reference_value": reference[metric],
            "predicted_value": prediction[metric],
            "error": error,
            "absolute_error": error.abs(),
            "error_unit": unit,
        }).reset_index())

        summary_rows.append({
            "model": model,
            "metric": metric,
            "unit": unit,
            "n": int(error.count()),
            "ME": error.mean(),
            "MAE": error.abs().mean(),
            "RMSE": np.sqrt((error ** 2).mean()),
        })

    errors = pd.concat(error_tables, ignore_index=True)
    errors = errors.sort_values(["image_id", "metric"])
    errors.to_csv(
        OUTPUT_DIR / f"{model}_errors.csv", index=False, encoding="utf-8-sig"
    )
    print(f"{model}: Errors saved")


simplebaseline_res50: Errors saved
hrnet_w32: Errors saved
litehrnet18: Errors saved
rtmpose_m: Errors saved
vitpose_b: Errors saved
mspn_4stage: Errors saved


In [3]:
summary = pd.DataFrame(summary_rows)
summary.to_csv(
    OUTPUT_DIR / "measurement_error_summary.csv",
    index=False, encoding="utf-8-sig"
)
display(summary)
print("Output directory:", OUTPUT_DIR)


,model,metric,unit,n,ME,MAE,RMSE
0,simplebaseline_res50,ca_deg,CA,1397,0.428036,4.466084,7.064414
1,simplebaseline_res50,hva_deg,HVA,1397,-0.182203,1.324762,1.715635
2,simplebaseline_res50,nmp_percent,NMP,1397,0.027715,0.622850,0.874967
3,simplebaseline_res50,si_percent,SI,1397,1.558468,9.369092,14.554237
4,simplebaseline_res50,csi_percent,CSI,1397,0.723990,5.129053,8.024285
5,hrnet_w32,ca_deg,CA,1397,1.131773,4.420780,7.198908
6,hrnet_w32,hva_deg,HVA,1397,0.145900,1.296524,1.734137
7,hrnet_w32,nmp_percent,NMP,1397,-0.027279,0.616677,0.878209
8,hrnet_w32,si_percent,SI,1397,0.185269,8.918697,14.343800
9,hrnet_w32,csi_percent,CSI,1397,0.253906,5.014031,7.964635


Output directory: ..\..\test_results\0_measurements\errors
